In [ ]:
source("helpers/init.r")

library(cowplot)

In [ ]:
cohorts    <- fread(paste0(READY_DATA, "cohorts.txt"))
plt_cohort <- cohorts %>% filter(!ref) %>% pull(cohort) %>% unique()
ref_cohort <- cohorts %>% filter(ref)  %>% pull(cohort) %>% unique()

# Validate primaryTumorLocation against available schemas
location <- cohorts %>% filter(!ref) %>% pull(primaryTumorLocation) %>% unique()
if (length(location) != 1) {
  stop("cohorts must have exactly one primaryTumorLocation for the test cohort (ref=FALSE).")
}
if (!location %in% names(schema_map)) {
  stop(paste0(
    "'", location, "' is not a valid primaryTumorLocation.
",
    "Valid options:
  ",
    paste(sort(names(schema_map)), collapse = "
  ")
  ))
}
message("✓ primaryTumorLocation '", location, "' is valid.")

# Enrich cohorts with doids from clinical if not already present
if (!"doids" %in% colnames(cohorts) || all(is.na(cohorts$doids)) || all(cohorts$doids == "")) {
  doids_df <- runQuery("clinical", cols = "sampleId, doids")
  cohorts  <- cohorts %>%
    select(-any_of("doids")) %>%
    left_join(doids_df %>% mutate(across(where(is.character), ~ na_if(., "NULL"))),
              by = "sampleId")
  fwrite(cohorts, paste0(READY_DATA, "cohorts.txt"))
}

# 0 - Load data

In [ ]:
clinical <- runQuery("clinical")
markers  <- runQuery("markers") %>% select(sampleId, rnaProliferation)

base <-
  cohorts %>%
  select(-any_of(c("doids", "primaryTumorLocation"))) %>%
  left_join(clinical, by = "sampleId") %>%
  left_join(markers,  by = "sampleId") %>%
  filter(cohort == plt_cohort) %>%
  mutate(across(where(is.character), ~ na_if(., "NULL")))

color_map        <- c(fills[["plt"]], fills[["ref"]])
names(color_map) <- c(plt_cohort, ref_cohort)

In [ ]:
# ── Oncotree enrichment ───────────────────────────────────────────────────────
# primaryTumorLocation comes from cohorts; primaryTumorType from clinical DB.
# oncotree column format is "Full Name (CODE)" — extract just the CODE.
oncotree_map <- fread(paste0(REFS_DIR, "oncotree.csv"))

test_samples <- cohorts %>% filter(!ref) %>% select(sampleId, primaryTumorLocation)

joined <- test_samples %>%
  left_join(clinical %>% select(sampleId, primaryTumorType), by = "sampleId") %>%
  left_join(oncotree_map, by = c("primaryTumorLocation", "primaryTumorType"))

n_matched <- sum(!is.na(joined$oncotree))
n_total   <- nrow(joined)
message(sprintf("Oncotree join: %d / %d samples matched", n_matched, n_total))
if (n_matched == 0) {
  message("  Locations in cohorts:     ", paste(unique(test_samples$primaryTumorLocation), collapse = ", "))
  message("  Locations in oncotree.csv:", paste(unique(oncotree_map$primaryTumorLocation), collapse = ", "))
  message("  Types in clinical:        ",
    paste(unique((clinical %>% filter(sampleId %in% test_samples$sampleId))$primaryTumorType), collapse = ", "))
  message("  Types in oncotree.csv:    ", paste(unique(oncotree_map$primaryTumorType), collapse = ", "))
}

# Extract code from "Full Name (CODE)" format
oncotree_codes <- joined %>%
  filter(!is.na(oncotree), oncotree != "") %>%
  mutate(code = stringr::str_extract(oncotree, "(?<=\\()\\w+(?=\\))")) %>%
  filter(!is.na(code)) %>%
  pull(code) %>%
  unique() %>%
  sort()

fwrite(data.frame(oncotree = oncotree_codes),
       paste0(READY_DATA, "oncotree_codes.txt"))
message("\u2713 Oncotree codes saved: ",
        if (length(oncotree_codes) > 0) paste(oncotree_codes, collapse = ", ") else "(none matched)")


In [ ]:
validate_columns(clinical, c("sampleId", "purpleQC", "biopsyType", "biopsyStructure",
                             "bestOverallResponse", "treatment",
                             "hasSystemicPreTreatment"), "clinical")

# 1 - Data availability

In [ ]:
avail_fields <- c("Response", "Treatment", "Pretreatment", "RNA", "Biopsy", "WGS")

clin_avail <-
  base %>%
  as_tibble() %>%
  transmute(
    sampleId,
    cohort,
    WGS             = as.integer(!is.na(purpleQC)),
    RNA             = as.integer(!is.na(rnaProliferation)),
    Biopsy          = as.integer(!is.na(biopsyType)),
    Response        = as.integer(!is.na(bestOverallResponse)),
    Treatment       = as.integer(!is.na(treatment)),
    Pretreatment = as.integer(!is.na(hasSystemicPreTreatment))
  ) %>%
  pivot_longer(-c(sampleId, cohort), names_to = "field", values_to = "val") %>%
  group_by(cohort, field) %>%
  summarise(avail = sum(val, na.rm = TRUE), .groups = "drop") %>%
  mutate(field = factor(field, levels = avail_fields),
         bar_alpha = ifelse(field %in% c("Treatment", "Response"), 0.5, 1.0),
         subtitle = "Number of patients with available data")

In [ ]:
avail_plt <-
  ggplot(clin_avail, aes(y = field, x = avail, alpha = bar_alpha)) +
  geom_bar(stat = "identity", position = "dodge", color = "black", fill = hartwig_red, linewidth = 0.3, width = 0.7) +
  geom_text(aes(label = avail), position = position_dodge(0.7), hjust = -0.2, size = 3.5) +
  scale_x_continuous(expand = expansion(mult = c(0, 0.2))) +
  scale_alpha_identity() +
  labs(title = "Data availability", subtitle = "Number of patients") +
  blue_subtitle +
  base_theme +
  theme(legend.position = "none", axis.title = element_blank(),
        axis.text.x = element_blank(), axis.ticks.x = element_blank(),
        plot.title = element_text(hjust = 0.5),
        strip.text = element_text(hjust = 0.5),
        plot.margin = margin(t = 5.5, r = 5.5, b = 5.5, l = 15))


# 2 - Treatment & response

In [ ]:
trt_levels <- rev(c("Chemo", "Hormonal", "Targeted", "Immuno", "Multiple", "Other", "Not available"))

trt_ready <-
  base %>%
  mutate(
    trt_cat = case_when(
      !is.na(treatment) & treatment != "NULL" & type == "Chemotherapy"                            ~ "Chemo",
      !is.na(treatment) & treatment != "NULL" & type %in% c("Androgen/estrogen deprivation therapy",
                                                             "Hormonal therapy")                  ~ "Hormonal",
      !is.na(treatment) & treatment != "NULL" & type %in% c("Targeted therapy",
                                                             "targeted therapy")                  ~ "Targeted",
      !is.na(treatment) & treatment != "NULL" & type == "Immunotherapy"                          ~ "Immuno",
      !is.na(treatment) & treatment != "NULL" & type == "Multiple therapy"                       ~ "Multiple",
      !is.na(treatment) & treatment != "NULL" & !is.na(type)                                     ~ "Other",
      TRUE                                                                                        ~ "Not available"
    ),
    trt_cat = factor(trt_cat, levels = trt_levels)
  ) %>%
  group_by(cohort, trt_cat) %>%
  summarise(
    `Post-biopsy treatment` = n(),
    Responders              = sum(as.numeric(bestOverallResponse), na.rm = TRUE),
    n_no_response           = sum(is.na(bestOverallResponse)),
    .groups = "drop"
  ) %>%
  mutate(pct = paste0(round(100 * Responders / `Post-biopsy treatment`), "%")) %>%
  pivot_longer(-c(cohort, trt_cat, pct, n_no_response), names_to = "field", values_to = "value") %>%
  mutate(
    label = case_when(
      field == "Responders" & trt_cat == "Not available"       ~ "N/A",
      field == "Responders" & n_no_response > 0                ~ paste0(value, " (", pct, ")  |  ", n_no_response, " response N/A"),
      field == "Responders"                                    ~ paste0(value, " (", pct, ")"),
      TRUE                                                      ~ as.character(value)
    ),
    label_x = ifelse(value == 0, 0.5, value * 1.05)
  )


In [ ]:
trt_plt <-
  ggplot(trt_ready, aes(y = trt_cat, x = value)) +
  geom_bar(stat = "identity", position = "dodge", color = "black", fill = hartwig_red, alpha = .5, linewidth = 0.3, width = 0.7) +
  geom_text(aes(label = label, x = label_x), hjust = 0, size = 3.5) +
  scale_x_continuous(expand = expansion(mult = c(0, 0.4))) +
  facet_wrap(~field, scales = "free_x") +
  labs(title = "Treatment response") +
  base_theme +
  theme(legend.position = "none", axis.title = element_blank(),
        axis.text.x = element_blank(), axis.ticks.x = element_blank(),
        plot.title = element_text(hjust = 0.5),
        strip.text = element_text(hjust = 0.5))


# 3 - Biopsy sites

In [ ]:
biopsy_ready <-
  base %>%
  drop_na(biopsyType) %>%
  mutate(
    loc = case_when(
      biopsyType == "Primary"                                                ~ "Primary",
      grepl("liver",               biopsyStructure, ignore.case = TRUE)  ~ "Liver",
      grepl("lymph.node",           biopsyStructure, ignore.case = TRUE)  ~ "Lymph",
      grepl("peritoneum|omentum",   biopsyStructure, ignore.case = TRUE)  ~ "Abdominal",
      grepl("^bone",                biopsyStructure, ignore.case = TRUE)  ~ "Bone",
      TRUE                                                                  ~ "Other"
    )
  ) %>%
  group_by(cohort, loc) %>%
  summarise(ct = n(), .groups = "drop") %>%
  group_by(cohort) %>%
  mutate(pct = ct / sum(ct)) %>%
  ungroup() %>%
  mutate(subtitle = "Biopsy location")


In [ ]:
biopsy_plt <- {
  # Order: Primary, then metastatic by frequency desc, Other last
  d <- biopsy_ready %>%
    filter(cohort == plt_cohort) %>%
    mutate(order_group = case_when(
      loc == "Primary" ~ 1L,
      loc == "Other"   ~ 3L,
      TRUE             ~ 2L
    )) %>%
    arrange(order_group, desc(ct)) %>%
    select(-order_group)

  loc_order <- as.character(d$loc)
  n_total   <- sum(d$ct)
  # Target ~2:1 width:height for a wider, more rectangular layout
  n_rows    <- max(2L, ceiling(sqrt(n_total / 2)))
  cols      <- ceiling(n_total / n_rows)

  # Flat top: row 1 fills completely; shorter row appears only at the bottom
  grid_df <- data.frame(
    loc = factor(rep(loc_order, d$ct), levels = loc_order)
  ) %>%
    mutate(
      x = (row_number() - 1L) %% cols + 1L,
      y = n_rows - (row_number() - 1L) %/% cols
    )

  n_primary    <- sum(d$ct[d$loc == "Primary"])
  n_metastasis <- n_total - n_primary
  count_label  <- paste0("Primary (n=", n_primary, ")  \u00b7  Metastasis (n=", n_metastasis, ")")

  ggplot(grid_df, aes(x, y, fill = loc)) +
    geom_tile(color = "white", linewidth = 0.5, alpha = 0.85) +
    scale_fill_manual(
      values = c(
        "Primary"    = hartwig_red,
        "Liver"      = "#e07b39",
        "Lymph" = "#4caf8b",
        "Abdominal"  = "#7b5ea7",
        "Bone"       = "#c9a96e",
        "Other"      = "#888888"
      ), name = NULL) +
    scale_x_continuous(expand = c(0, 0)) +
    scale_y_continuous(expand = c(0, 0)) +
    coord_equal() +
    labs(title = "Biopsy type", subtitle = count_label) +
    base_theme +
    theme(axis.text = element_blank(), axis.ticks = element_blank(),
          axis.title = element_blank(), panel.grid = element_blank(),
          axis.line = element_blank(), legend.position = "bottom",
          legend.direction = "horizontal",
          legend.text      = element_text(size = rel(0.8)),
          plot.title    = element_text(hjust = 0.5),
          plot.subtitle = element_text(hjust = 0.5, size = 9, color = "grey40"),
          aspect.ratio  = n_rows / cols)
}


# 5 - Save figures

In [ ]:
# ── Assemble ──────────────────────────────────────────────────────────────────
clinical_figure <- plot_grid(
  plot_grid(avail_plt, trt_plt, biopsy_plt, nrow = 1, rel_widths = c(.9, 1.3, .7))
)

options(repr.plot.width = 13, repr.plot.height = 3.5)
clinical_figure

figures <- list(
  avail   = avail_plt,
  trt     = trt_plt,
  biopsy  = biopsy_plt,
  figure  = clinical_figure
)
saveRDS(figures, paste0(READY_FIG, "clinical.rds"))